In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/hbdz936@gmail.com/consolidated_pipeline/setup/utilities

In [0]:
print(bronze_schema, silver_schema, gold_schema)

bronze silver gold


In [0]:
# Bypass widgets entirely for now
catalog = "fmcg"
bronze_schema = "bronze"
silver_schema = "silver"
gold_schema = "gold"
data_source = "orders"  # hardcoded

bronze_table = f"{catalog}.{bronze_schema}.{data_source}"
silver_table = f"{catalog}.{silver_schema}.{data_source}"
gold_table = f"{catalog}.{gold_schema}.sb_fact_{data_source}"

print(f"Writing to: {bronze_table}")  # Must print fmcg.bronze.orders
assert bronze_table == "fmcg.bronze.orders", "WRONG TABLE — STOP"

base_path = "/Volumes/fmcg/bronze/sports_bar_raw/orders"
landing_path = "/Volumes/fmcg/bronze/sports_bar_raw/orders/landing/"
processed_path = "/Volumes/fmcg/bronze/sports_bar_raw/orders/processed/"
print("Base Path: ", base_path)
print("Landing Path: ", landing_path)
print("Processed Path: ", processed_path)


# define the tables
bronze_table = f"{catalog}.{bronze_schema}.{data_source}"
silver_table = f"{catalog}.{silver_schema}.{data_source}"
gold_table = f"{catalog}.{gold_schema}.sb_fact_{data_source}"

Writing to: fmcg.bronze.orders
Base Path:  /Volumes/fmcg/bronze/sports_bar_raw/orders
Landing Path:  /Volumes/fmcg/bronze/sports_bar_raw/orders/landing/
Processed Path:  /Volumes/fmcg/bronze/sports_bar_raw/orders/processed/


In [0]:
df = spark.read.options(header=True, inferSchema=True).csv(f"{landing_path}/*.csv").withColumn("read_timestamp", F.current_timestamp()).select("*", "_metadata.file_name", "_metadata.file_size")

print("Total Rows: ", df.count())
df.show(5)

Total Rows:  51810
+------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
|    order_id|order_placement_date|customer_id|product_id|order_qty|      read_timestamp|           file_name|file_size|
+------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
|FOCT62720602|Tuesday, Septembe...|     ABC987|  25891301|     71.0|2026-04-06 12:21:...|orders_2025_09_30...|    41446|
|FOCT62720602|Tuesday, Septembe...|     789720|  25891502|    125.0|2026-04-06 12:21:...|orders_2025_09_30...|    41446|
|FOCT62720602|Tuesday, Septembe...|     789720|  25891403|    462.0|2026-04-06 12:21:...|orders_2025_09_30...|    41446|
|FOCT62720602|Tuesday, Septembe...|    INVALID|  25891601|    133.0|2026-04-06 12:21:...|orders_2025_09_30...|    41446|
|FOCT62720602|Tuesday, Septembe...|     789720|  25891602|     79.0|2026-04-06 12:21:...|orders_2025_09_30...|    41446|
+------------

In [0]:
display(df.limit(20))

order_id,order_placement_date,customer_id,product_id,order_qty,read_timestamp,file_name,file_size
FOCT62720602,"Tuesday, September 30, 2025",ABC987,25891301,71.0,2026-04-06T12:22:30.436Z,orders_2025_09_30.csv,41446
FOCT62720602,"Tuesday, September 30, 2025",789720,25891502,125.0,2026-04-06T12:22:30.436Z,orders_2025_09_30.csv,41446
FOCT62720602,"Tuesday, September 30, 2025",789720,25891403,462.0,2026-04-06T12:22:30.436Z,orders_2025_09_30.csv,41446
FOCT62720602,"Tuesday, September 30, 2025",INVALID,25891601,133.0,2026-04-06T12:22:30.436Z,orders_2025_09_30.csv,41446
FOCT62720602,"Tuesday, September 30, 2025",789720,25891602,79.0,2026-04-06T12:22:30.436Z,orders_2025_09_30.csv,41446
FOCT62720602,2025/09/30,XYZ123,25891101,472.0,2026-04-06T12:22:30.436Z,orders_2025_09_30.csv,41446
FOCT62720602,2025/09/30,ABC987,25891301,71.0,2026-04-06T12:22:30.436Z,orders_2025_09_30.csv,41446
FOCT62402303,"Tuesday, September 30, 2025",789402,25891303,24.0,2026-04-06T12:22:30.436Z,orders_2025_09_30.csv,41446
FOCT62402303,"Tuesday, September 30, 2025",789402,25891301,26.0,2026-04-06T12:22:30.436Z,orders_2025_09_30.csv,41446
FOCT62402303,"Tuesday, September 30, 2025",789402,25891102,328.0,2026-04-06T12:22:30.436Z,orders_2025_09_30.csv,41446


In [0]:
df = df.withColumn("product_id", F.col("product_id").cast("string"))

In [0]:
df = spark.read.options(header=True, inferSchema=True)\
    .csv(f"{landing_path}/*.csv")\
    .withColumn("read_timestamp", F.current_timestamp())\
    .select("*", "_metadata.file_name", "_metadata.file_size")

df = df.withColumn("product_id", F.col("product_id").cast("string"))

print(f"Row count: {df.count()}")
print(f"Target: {bronze_table}")
assert bronze_table == "fmcg.bronze.orders", "WRONG TABLE — STOP"

df.write \
    .format("delta") \
    .option("delta.enableChangeDataFeed", "true") \
    .option("mergeSchema", "true") \
    .mode("append") \
    .saveAsTable(bronze_table)

print("Write complete. Verifying...")
print(spark.table(bronze_table).count())

Row count: 51810
Target: fmcg.bronze.orders
Write complete. Verifying...
51810


In [0]:
files = dbutils.fs.ls(landing_path)
for file_info in files:
    dbutils.fs.mv(
        file_info.path,
        f"{processed_path}/{file_info.name}",
        True
    )

In [0]:
df_orders = spark.sql(f"SELECT * FROM {bronze_table}")
df_orders.show(2)

+------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
|    order_id|order_placement_date|customer_id|product_id|order_qty|      read_timestamp|           file_name|file_size|
+------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
|FJUL33320501|          2025/07/01|     789320|  25891203|    150.0|2026-04-06 12:23:...|orders_2025_07_01...|    20744|
|FJUL33320501|          2025/07/01|     789320|  25891301|     46.0|2026-04-06 12:23:...|orders_2025_07_01...|    20744|
+------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
only showing top 2 rows


In [0]:
# 1. Keep only rows where order_qty is present
df_orders = df_orders.filter(F.col("order_qty").isNotNull())


# 2. Clean customer_id → keep numeric, else set to 999999
df_orders = df_orders.withColumn(
    "customer_id",
    F.when(F.col("customer_id").rlike("^[0-9]+$"), F.col("customer_id"))
     .otherwise("999999")
     .cast("string")
)

# 3. Remove weekday name from the date text
#    "Tuesday, July 01, 2025" → "July 01, 2025"
df_orders = df_orders.withColumn(
    "order_placement_date",
    F.regexp_replace(F.col("order_placement_date"), r"^[A-Za-z]+,\s*", "")
)

# 4. Parse order_placement_date using multiple possible formats
df_orders = df_orders.withColumn(
    "order_placement_date",
    F.coalesce(
        F.try_to_date("order_placement_date", "yyyy/MM/dd"),
        F.try_to_date("order_placement_date", "dd-MM-yyyy"),
        F.try_to_date("order_placement_date", "dd/MM/yyyy"),
        F.try_to_date("order_placement_date", "MMMM dd, yyyy"),
    )
)

# 5. Drop duplicates
df_orders = df_orders.dropDuplicates(["order_id", "order_placement_date", "customer_id", "product_id", "order_qty"])

# 5. convert product id to string
df_orders = df_orders.withColumn('product_id', F.col('product_id').cast('string'))

In [0]:
# check what's the maximum and minimum date
df_orders.agg(
    F.min("order_placement_date").alias("min_date"),
    F.max("order_placement_date").alias("max_date")
).show()

+----------+----------+
|  min_date|  max_date|
+----------+----------+
|2025-07-01|2025-11-30|
+----------+----------+



In [0]:
#JOIN WITH PRODUCTS
df_products = spark.table("fmcg.silver.products")
df_joined = df_orders.join(df_products, on="product_id", how="inner").select(df_orders["*"], df_products["product_code"])

df_joined.show(5)

+------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+--------------------+
|    order_id|order_placement_date|customer_id|product_id|order_qty|      read_timestamp|           file_name|file_size|        product_code|
+------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+--------------------+
|FJUL33320501|          2025-07-01|     789320|  25891201|    354.0|2026-04-06 12:23:...|orders_2025_07_01...|    20744|2e387cef1424d6e7b...|
|FJUL32221603|          2025-07-01|     789221|  25891501|    140.0|2026-04-06 12:23:...|orders_2025_07_01...|    20744|ee1f7df9cf660ef02...|
|FJUL34303602|          2025-07-01|     789303|  25891202|    144.0|2026-04-06 12:23:...|orders_2025_07_01...|    20744|0cb7b2f42657b625f...|
|FJUL34903603|          2025-07-01|     789903|  25891603|    133.0|2026-04-06 12:23:...|orders_2025_07_01...|    20744|451f7167b28a25bde...|
|FJUL3

In [0]:
#UPSERT TO SILVER TABLE
if not (spark.catalog.tableExists(silver_table)):
    df_joined.write.format("delta").option(
        "delta.enableChangeDataFeed", "true"
    ).option("mergeSchema", "true").mode("overwrite").saveAsTable(silver_table)
else:
    silver_delta = DeltaTable.forName(spark, silver_table)
    silver_delta.alias("silver").merge(df_joined.alias("bronze"), "silver.order_placement_date = bronze.order_placement_date AND silver.order_id = bronze.order_id AND silver.product_code = bronze.product_code AND silver.customer_id = bronze.customer_id").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

### GOLD

In [0]:
df_gold = spark.sql(f"SELECT order_id, order_placement_date as date, customer_id as customer_code, product_code, product_id, order_qty as sold_quantity FROM {silver_table};")

df_gold.show(2)

+------------+----------+-------------+--------------------+----------+-------------+
|    order_id|      date|customer_code|        product_code|product_id|sold_quantity|
+------------+----------+-------------+--------------------+----------+-------------+
|FJUL33320501|2025-07-01|       789320|2e387cef1424d6e7b...|  25891201|        354.0|
|FJUL32221603|2025-07-01|       789221|ee1f7df9cf660ef02...|  25891501|        140.0|
+------------+----------+-------------+--------------------+----------+-------------+
only showing top 2 rows


In [0]:
#NOTHING SPECIAL IN GOLD TABLE SINCE WE ARE NOT DOING ANY CALCULATIONS ETC, EG. GIVEN PROFIT AND WE CALC REVENUE
if not (spark.catalog.tableExists(gold_table)):
    print("creating New Table")
    df_gold.write.format("delta").option(
        "delta.enableChangeDataFeed", "true"
    ).option("mergeSchema", "true").mode("overwrite").saveAsTable(gold_table)
else:
    gold_delta = DeltaTable.forName(spark, gold_table)
    gold_delta.alias("source").merge(df_gold.alias("gold"), "source.date = gold.date AND source.order_id = gold.order_id AND source.product_code = gold.product_code AND source.customer_code = gold.customer_code").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

creating New Table


### MERGE WITH PARENT COMPANY

In [0]:
#SOLD QUANTITY IS ON BASIS OF MONTHS IN PARENT COMPANY 
# IN CHILD COMPANY IT IS ON BASIS OF DATES, DAILY

In [0]:
df_child = spark.sql(f"SELECT date, product_code, customer_code, sold_quantity FROM {gold_table}")
df_child.show(10)

+----------+--------------------+-------------+-------------+
|      date|        product_code|customer_code|sold_quantity|
+----------+--------------------+-------------+-------------+
|2025-07-01|2e387cef1424d6e7b...|       789320|        354.0|
|2025-07-01|ee1f7df9cf660ef02...|       789221|        140.0|
|2025-07-01|0cb7b2f42657b625f...|       789303|        144.0|
|2025-07-01|451f7167b28a25bde...|       789903|        133.0|
|2025-07-01|77b6f538a9d0e0cf8...|       789220|        372.0|
|2025-07-01|102628255d24304d6...|       789420|        357.0|
|2025-07-01|0cb7b2f42657b625f...|       789421|        332.0|
|2025-07-01|451f7167b28a25bde...|       789703|        165.0|
|2025-07-01|102628255d24304d6...|       789420|        325.0|
|2025-07-01|062f5574bbdf4386b...|       789221|        101.0|
+----------+--------------------+-------------+-------------+
only showing top 10 rows


In [0]:
#FIRST CHANGE THE DATE OF PARENT COMPANY TO FIRSTY DAY OF MONTH. Eg. 2025-07-10 -> 2025-07-1

df_monthly = (
    df_child
    # 1. Get month start date (e.g., 2025-11-30 → 2025-11-01)
    .withColumn("month_start", F.trunc("date", "MM"))   # or F.date_trunc("month", "date").cast("date")

    # 2.Group at monthly grain by month_start + product_code + customer_code
    .groupBy("month_start", "product_code", "customer_code")
    .agg(
        F.sum("sold_quantity").alias("sold_quantity")
    )

    # 3. Rename month_start back to `date` to match your target schema
    .withColumnRenamed("month_start", "date")
)

df_monthly.show(5, truncate=False)

+----------+----------------------------------------------------------------+-------------+-------------+
|date      |product_code                                                    |customer_code|sold_quantity|
+----------+----------------------------------------------------------------+-------------+-------------+
|2025-07-01|2e387cef1424d6e7b162b45622d4b1a788d11776e33d05cc8552f4ecd2ea1896|789320       |2735.0       |
|2025-07-01|ee1f7df9cf660ef02c33037d8d6eb94cbefe8e7b84c306e9387f09b0cae0abae|789221       |3014.0       |
|2025-07-01|0cb7b2f42657b625f754e833aa1cf6a967be26f17415f5342302ebb0e90c8a28|789303       |3223.0       |
|2025-07-01|451f7167b28a25bde73995910e31c07dfa26411f1db47847f19e16747effbdaa|789903       |1558.0       |
|2025-07-01|77b6f538a9d0e0cf845db5c2cbecec46fdd30303b501e06f64baf1d4dc0e66f9|789220       |4498.0       |
+----------+----------------------------------------------------------------+-------------+-------------+
only showing top 5 rows


In [0]:
gold_parent_delta = DeltaTable.forName(spark, f"{catalog}.{gold_schema}.fact_orders")
gold_parent_delta.alias("parent_gold").merge(df_monthly.alias("child_gold"), "parent_gold.date = child_gold.date AND parent_gold.product_code = child_gold.product_code AND parent_gold.customer_code = child_gold.customer_code").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]